# Validação — antes/depois na amostra (S18)

Protocolo e plano de análise fixados no **ADR 004** (`docs/decisoes/004-amostra-e-protocolo-da-validacao.md`) antes das medições.

- **Entrada principal:** `data/processed/validation-v1.0.csv`, a rodada da tag `v1.0` com o protocolo fixado antes das medições (cópia do `validation.csv` commitado na `v1.0`). Uma linha por site, com original, corrigido e delta de cada desfecho.
- **Entrada posterior (seção 5):** `data/processed/validation.csv`, a rodada com as regras revistas depois de ver o resultado da `v1.0` (guarda de imagens na `img-dimensions`, adendo ao ADR 014; `landmarks-v2`, adendo ao ADR 016). Mesma amostra (`data/validation/amostra.csv`) e mesmo protocolo. **Análise posterior, não confirmatória.**
- **Entrada do censo (seção 6):** `data/processed/validation-censo.csv`, todos os elegíveis do ADR 004 menos 14 fora do escopo (134 sites, `data/validation/censo.csv`), com as regras da S18b. Pré-registrado no adendo do censo ao ADR 004 antes das medições. **Análise secundária, não confirmatória.**
- **Comparação:** cópia local original × cópia local corrigida (regra 7, ADR 018), mesmo servidor, Lighthouse 3× por variante (mediana pela nota, ADR 009). Delta = corrigido − original.
- **Teste:** Wilcoxon pareado (*signed-rank*), bilateral, α = 0,05; Holm na família principal (nota de performance, LCP, CLS, TBT, índice de semântica). Pares com delta 0 saem do teste (`zero_method="wilcox"`) e o n efetivo é informado. Efeito: mediana das diferenças e correlação rank-bisserial pareada, r = (W⁺ − W⁻) / (W⁺ + W⁻).
- **Saídas:** `docs/resultados/tabelas/validacao-*.csv`, `docs/resultados/figuras/validacao-antes-depois.png` e `docs/resultados/tabelas/validacao-meta.json` (regra 10).
- n = 12: resultados exploratórios (ADR 004, limitações).

Executar da raiz do repo: `python -m jupyter nbconvert --to notebook --execute --inplace analysis/validacao.ipynb`.

In [1]:
import hashlib, json, platform, subprocess
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
VALIDACAO = ROOT / "data" / "processed" / "validation-v1.0.csv"
POSTERIOR = ROOT / "data" / "processed" / "validation.csv"
AMOSTRA = ROOT / "data" / "validation" / "amostra.csv"
CENSO = ROOT / "data" / "processed" / "validation-censo.csv"
LISTA_CENSO = ROOT / "data" / "validation" / "censo.csv"
TAB = ROOT / "docs" / "resultados" / "tabelas"
FIG = ROOT / "docs" / "resultados" / "figuras"
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

ALFA = 0.05
# ADR 004, 7: família principal (Holm). Sentido de "melhor" de cada desfecho.
PRINCIPAIS = {"nota_performance": "maior", "lcp_ms": "menor", "cls": "menor", "tbt_ms": "menor", "indice_semantica": "maior"}

def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None

sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
meta = {
    "generatedAt": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "toolVersions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "scipy": scipy.__version__, "matplotlib": matplotlib.__version__},
    "gitCommit": git("rev-parse", "HEAD"),
    "entrada": {
        "validation_v1_0": {"arquivo": "data/processed/validation-v1.0.csv", "sha256": sha(VALIDACAO)},
        "validation_posterior": {"arquivo": "data/processed/validation.csv", "sha256": sha(POSTERIOR)},
        "amostra": {"arquivo": "data/validation/amostra.csv", "sha256": sha(AMOSTRA)},
        "validation_censo": {"arquivo": "data/processed/validation-censo.csv", "sha256": sha(CENSO)},
        "censo": {"arquivo": "data/validation/censo.csv", "sha256": sha(LISTA_CENSO)},
    },
}
meta

{'generatedAt': '2026-11-16T23:30:51+00:00',
 'toolVersions': {'python': '3.13.5',
  'pandas': '2.3.1',
  'numpy': '2.2.6',
  'scipy': '1.16.3',
  'matplotlib': '3.10.5'},
 'gitCommit': '1b072fddeed1953cdd999d862b36f7b731a4e26e',
 'entrada': {'validation_v1_0': {'arquivo': 'data/processed/validation-v1.0.csv',
   'sha256': 'e120477b686a19edbf9302ac8173614e39ca3f58d0a12e7fa370a04856d9c98f'},
  'validation_posterior': {'arquivo': 'data/processed/validation.csv',
   'sha256': 'e8c451e6c728603416a47d09401a84efc7e1757eaab326590bf0d50e8551fe6c'},
  'amostra': {'arquivo': 'data/validation/amostra.csv',
   'sha256': '04f408ebd7957f15acb288ace768d917965403ef3c0a8acbfbcd15a2fa5fb663'},
  'validation_censo': {'arquivo': 'data/processed/validation-censo.csv',
   'sha256': '4d7071bb204d17cbabfd12f19f0dd1f8822d7e9a1091f973df6690560c5a09f2'},
  'censo': {'arquivo': 'data/validation/censo.csv',
   'sha256': '7bd97a01b70858929ccdecebe0f43ad908b14a42346e3ab348c220187323c218'}}}

In [2]:
v = pd.read_csv(VALIDACAO)
amostra = pd.read_csv(AMOSTRA)
faltando = sorted(set(amostra.id) - set(v.id))
print("sites na amostra:", len(amostra), "| validados:", len(v), "| faltando:", faltando)
cols = ["id", "grupo", "correcoes", "correcoes_llm", "faltas_original", "faltas_corrigido", "amplitude_original", "amplitude_corrigido",
        "nota_performance_original", "nota_performance_corrigido", "nota_performance_delta",
        "indice_semantica_original", "indice_semantica_corrigido", "indice_semantica_delta"]
por_site = v[cols].sort_values(["grupo", "id"]).reset_index(drop=True)
por_site.to_csv(TAB / "validacao-por-site.csv", index=False)
por_site

sites na amostra: 12 | validados: 12 | faltando: []


,id,grupo,correcoes,correcoes_llm,faltas_original,faltas_corrigido,amplitude_original,amplitude_corrigido,nota_performance_original,nota_performance_corrigido,nota_performance_delta,indice_semantica_original,indice_semantica_corrigido,indice_semantica_delta
0,ce.ufrn.br,estatico,20,0,15,15,19,14,41,35,-6,59.007817,59.007817,0.000000
1,espacos-denf.imd.ufrn.br,estatico,4,0,2,2,0,16,42,42,0,37.593985,42.857143,5.263158
2,materialpublic.imd.ufrn.br,estatico,8,2,2,2,2,1,67,68,1,37.429775,56.460674,19.030899
3,agir.ufrn.br,framework-js,136,0,39,39,4,1,57,58,1,49.644719,54.907877,5.263158
4,entbl2020.ccsa.ufrn.br,framework-js,1,0,8,8,0,0,45,45,0,57.943245,63.206403,5.263158
5,semanadeturismo.ccsa.ufrn.br,framework-js,1,0,8,8,3,1,48,48,0,46.597714,51.860872,5.263158
6,observatoriodasdesigualdades.ccsa.ufrn.br,outro,9,0,64,64,10,6,81,81,0,73.419878,73.419878,0.000000
7,periodicos.ufrn.br,outro,36,0,12,12,3,1,65,66,1,60.441086,60.441086,0.000000
8,ria.ufrn.br,outro,14,0,3,3,3,7,84,85,1,61.729906,66.993064,5.263158
9,3cnienssup.ufrn.br,wordpress,4,0,33,33,0,0,48,67,19,60.352891,60.352891,0.000000


## 1. Teste pareado na família principal

Para LCP, CLS e TBT, valores menores são melhores: um delta negativo é melhora.

In [3]:
def holm(p):
    p = np.asarray(p, dtype=float)
    ordem = np.argsort(p)
    m = len(p)
    aj = np.empty(m)
    acumulado = 0.0
    for k, i in enumerate(ordem):
        acumulado = max(acumulado, min(1.0, (m - k) * p[i]))
        aj[i] = acumulado
    return aj

def wilcoxon_pareado(delta):
    d = pd.Series(delta).dropna()
    nz = d[d != 0]
    linha = {"n_pares": len(d), "n_efetivo": len(nz), "melhorou": None, "piorou": None, "iguais": int((d == 0).sum()),
             "mediana_delta": float(d.median()) if len(d) else np.nan, "W": np.nan, "p": np.nan, "r_rank_bisserial": np.nan}
    if len(nz) == 0:
        return linha
    postos = stats.rankdata(nz.abs())
    w_mais, w_menos = postos[nz > 0].sum(), postos[nz < 0].sum()
    res = stats.wilcoxon(nz, zero_method="wilcox", alternative="two-sided")
    linha.update(W=float(res.statistic), p=float(res.pvalue), r_rank_bisserial=float((w_mais - w_menos) / (w_mais + w_menos)))
    return linha

def familia(v):
    linhas = []
    for desf, melhor in PRINCIPAIS.items():
        d = v[f"{desf}_delta"]
        r = wilcoxon_pareado(d)
        sinal = 1 if melhor == "maior" else -1
        r["melhorou"] = int((sinal * d > 0).sum())
        r["piorou"] = int((sinal * d < 0).sum())
        linhas.append({"desfecho": desf, "melhor_se": melhor,
                       "mediana_original": float(v[f"{desf}_original"].median()), "mediana_corrigido": float(v[f"{desf}_corrigido"].median()), **r})
    t = pd.DataFrame(linhas)
    testaveis = t.p.notna()
    t["p_holm"] = np.nan
    t.loc[testaveis, "p_holm"] = holm(t.loc[testaveis, "p"])
    t["significativo"] = t.p_holm < ALFA
    return t

principal = familia(v)
principal.to_csv(TAB / "validacao-wilcoxon.csv", index=False)
principal.round(4)

,desfecho,melhor_se,mediana_original,mediana_corrigido,n_pares,n_efetivo,melhorou,piorou,iguais,mediana_delta,W,p,r_rank_bisserial,p_holm,significativo
0,nota_performance,maior,54.0000,61.5000,12,7,5,2,5,0.0000,11.0,0.6875,0.2143,1.0000,False
1,lcp_ms,menor,9547.8050,9613.8394,12,12,8,4,0,-8.0492,28.0,0.4238,-0.2821,1.0000,False
2,cls,menor,0.0013,0.0005,12,5,4,1,7,0.0000,3.0,0.3125,-0.6000,1.0000,False
3,tbt_ms,menor,115.5000,118.7500,12,11,6,5,1,-0.1657,31.0,0.8799,0.0606,1.0000,False
4,indice_semantica,maior,59.1331,59.8056,12,7,6,1,5,2.6316,1.0,0.0312,0.9286,0.1562,False


## 2. Descritivas dos demais desfechos

Sem teste (ADR 004, 7): outras notas do Lighthouse, métricas de carga e as 8 subnotas do índice.

In [4]:
outros = [c[: -len("_delta")] for c in v.columns if c.endswith("_delta") and c[: -len("_delta")] not in PRINCIPAIS]
desc = pd.DataFrame([{
    "desfecho": k,
    "n": int(v[f"{k}_delta"].notna().sum()),
    "mediana_original": v[f"{k}_original"].median(),
    "mediana_corrigido": v[f"{k}_corrigido"].median(),
    "mediana_delta": v[f"{k}_delta"].median(),
    "delta_positivo": int((v[f"{k}_delta"] > 0).sum()),
    "delta_negativo": int((v[f"{k}_delta"] < 0).sum()),
    "delta_zero": int((v[f"{k}_delta"] == 0).sum()),
} for k in outros])
desc.to_csv(TAB / "validacao-descritivas.csv", index=False)
desc.round(3)

C:\Users\Mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\Mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\Mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,desfecho,n,mediana_original,mediana_corrigido,mediana_delta,delta_positivo,delta_negativo,delta_zero
0,nota_accessibility,12,77.500,83.000,2.500,6,0,6
1,nota_best_practices,12,92.000,88.000,0.000,0,4,8
2,nota_seo,12,82.500,82.500,0.000,0,0,12
3,fcp_ms,12,4168.058,4172.994,-0.708,5,7,0
4,speed_index_ms,12,4820.603,4734.671,-0.708,5,7,0
5,peso_total_bytes,12,1519102.500,1519136.000,9.500,7,5,0
6,requisicoes,12,49.000,49.500,0.000,1,5,6
7,indice_semantica_pesos_iguais,12,64.702,65.743,3.571,6,1,5
8,sub_landmarks,12,75.000,75.000,0.000,1,0,11
9,sub_titulos,12,50.000,50.000,0.000,0,0,12


## 3. Delta da nota de performance × variação entre execuções

A amplitude (máx − mín das 3 execuções de uma variante) é o ruído do Lighthouse no próprio site. Um delta menor que a amplitude não se distingue da variação entre execuções.

In [5]:
ruido = v[["id", "nota_performance_delta", "amplitude_original", "amplitude_corrigido"]].copy()
ruido["amplitude_max"] = ruido[["amplitude_original", "amplitude_corrigido"]].max(axis=1)
ruido["delta_acima_do_ruido"] = ruido.nota_performance_delta.abs() > ruido.amplitude_max
print("sites com |delta| acima da maior amplitude:", int(ruido.delta_acima_do_ruido.sum()), "de", len(ruido))
ruido.sort_values("nota_performance_delta")

sites com |delta| acima da maior amplitude: 1 de 12


,id,nota_performance_delta,amplitude_original,amplitude_corrigido,amplitude_max,delta_acima_do_ruido
1,ce.ufrn.br,-6,19,14,19,False
7,profletras.ufrn.br,-2,2,2,2,False
3,semanadeturismo.ccsa.ufrn.br,0,3,1,3,False
0,espacos-denf.imd.ufrn.br,0,0,16,16,False
5,entbl2020.ccsa.ufrn.br,0,0,0,0,False
8,mat.ufrn.br,0,1,3,3,False
10,observatoriodasdesigualdades.ccsa.ufrn.br,0,10,6,10,False
2,materialpublic.imd.ufrn.br,1,2,1,2,False
9,ria.ufrn.br,1,3,7,7,False
4,agir.ufrn.br,1,4,1,4,False


## 4. Figura antes/depois

Uma linha por site, da cópia original à corrigida. Azul: melhorou; laranja: piorou; cinza: igual.

In [6]:
COR = {"melhorou": "#2a78d6", "piorou": "#eb6834", "igual": "#8a8985"}
PAINEIS = [("nota_performance", "Nota de performance (Lighthouse)"), ("indice_semantica", "Índice de semântica (ADR 003)")]
from matplotlib.lines import Line2D

def figura(v, titulo, arquivo):
    fig, eixos = plt.subplots(1, 2, figsize=(9, 4.6))
    for ax, (desf, tit) in zip(eixos, PAINEIS):
        for _, s in v.iterrows():
            o, c = s[f"{desf}_original"], s[f"{desf}_corrigido"]
            if pd.isna(o) or pd.isna(c):
                continue
            estado = "igual" if c == o else ("melhorou" if c > o else "piorou")
            ax.plot([0, 1], [o, c], color=COR[estado], linewidth=2, marker="o", markersize=5, alpha=0.9)
        ax.set_xticks([0, 1], ["Original", "Corrigido"])
        ax.set_xlim(-0.25, 1.25)
        ax.set_ylim(0, 100)
        ax.set_title(tit, fontsize=11)
        ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
    fig.legend(handles=[Line2D([0], [0], color=COR[k], linewidth=2, label=k) for k in COR], loc="lower center", ncol=3, frameon=False)
    fig.suptitle(titulo, fontsize=12)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.savefig(FIG / arquivo, dpi=150)
    plt.close(fig)
    print("figura:", FIG / arquivo)

figura(v, f"Cópia local original × corrigida (v1.0, n = {len(v)} sites, ADR 004)", "validacao-antes-depois.png")

figura: C:\Users\Mateo\Documents\tcc-ufrn-web-audit\docs\resultados\figuras\validacao-antes-depois.png


## 5. Análise posterior: regras revistas depois da v1.0

**Não confirmatória.** As regras foram mudadas depois de ver o resultado da `v1.0`: a guarda de renderização desfaz as correções da `img-dimensions` que mudam o tamanho exibido da imagem (adendo ao ADR 014), e o prompt `landmarks-v2` aceita `section`/`article` como candidatos (adendo ao ADR 016). Mesma amostra, mesmo protocolo e mesma família de testes. A seção mostra o efeito da revisão; o resultado principal continua sendo o da seção 1.

In [7]:
p = pd.read_csv(POSTERIOR)
assert sorted(p.id) == sorted(v.id), "a rodada posterior precisa ter os mesmos sites"
posterior = familia(p)
posterior.to_csv(TAB / "validacao-wilcoxon-posterior.csv", index=False)
posterior.round(4)

,desfecho,melhor_se,mediana_original,mediana_corrigido,n_pares,n_efetivo,melhorou,piorou,iguais,mediana_delta,W,p,r_rank_bisserial,p_holm,significativo
0,nota_performance,maior,52.0000,59.0000,12,5,4,1,7,0.0000,1.5,0.1875,0.8000,0.4424,False
1,lcp_ms,menor,9701.9663,9805.7028,12,12,10,2,0,-292.5627,11.0,0.0269,-0.7179,0.1074,False
2,cls,menor,0.0013,0.0049,12,5,3,2,7,0.0000,7.0,1.0000,-0.0667,1.0000,False
3,tbt_ms,menor,125.7500,81.5000,12,11,8,3,1,-11.6398,16.0,0.1475,-0.5152,0.4424,False
4,indice_semantica,maior,59.1331,59.8497,12,8,7,1,4,5.2632,1.0,0.0156,0.9444,0.0781,False


In [8]:
cmp = v[["id"]].copy()
for k in ["nota_performance", "nota_best_practices", "nota_accessibility", "cls", "indice_semantica", "sub_landmarks"]:
    cmp = cmp.merge(v[["id", f"{k}_delta"]].rename(columns={f"{k}_delta": f"{k}_v1_0"}), on="id")
    cmp = cmp.merge(p[["id", f"{k}_delta"]].rename(columns={f"{k}_delta": f"{k}_posterior"}), on="id")
cmp = cmp.merge(v[["id", "correcoes", "correcoes_llm"]].rename(columns={"correcoes": "correcoes_v1_0", "correcoes_llm": "correcoes_llm_v1_0"}), on="id")
cmp = cmp.merge(p[["id", "correcoes", "correcoes_llm"]].rename(columns={"correcoes": "correcoes_posterior", "correcoes_llm": "correcoes_llm_posterior"}), on="id")
cmp.to_csv(TAB / "validacao-v1-0-x-posterior.csv", index=False)
print("sites com boas práticas pior no corrigido: v1.0", int((v.nota_best_practices_delta < 0).sum()), "| posterior", int((p.nota_best_practices_delta < 0).sum()))
print("sites com correção da LLM: v1.0", int((v.correcoes_llm > 0).sum()), "| posterior", int((p.correcoes_llm > 0).sum()))
cmp.round(3)

sites com boas práticas pior no corrigido: v1.0 4 | posterior 0
sites com correção da LLM: v1.0 1 | posterior 3


,id,nota_performance_v1_0,nota_performance_posterior,nota_best_practices_v1_0,nota_best_practices_posterior,nota_accessibility_v1_0,nota_accessibility_posterior,cls_v1_0,cls_posterior,indice_semantica_v1_0,indice_semantica_posterior,sub_landmarks_v1_0,sub_landmarks_posterior,correcoes_v1_0,correcoes_llm_v1_0,correcoes_posterior,correcoes_llm_posterior
0,espacos-denf.imd.ufrn.br,0,0,-3,0,5,6,0.000,0.000,5.263,10.526,0,25,4,0,3,1
1,ce.ufrn.br,-6,2,0,0,0,0,-0.000,0.000,0.000,0.000,0,0,20,0,20,0
2,materialpublic.imd.ufrn.br,1,1,0,0,10,10,-0.018,-0.003,19.031,19.031,50,50,8,2,5,2
3,semanadeturismo.ccsa.ufrn.br,0,0,0,0,7,7,0.000,0.000,5.263,5.263,0,0,1,0,1,0
4,agir.ufrn.br,1,-1,-4,0,5,5,-0.001,-0.001,5.263,5.263,0,0,136,0,128,0
5,entbl2020.ccsa.ufrn.br,0,0,0,0,11,11,0.000,0.000,5.263,5.263,0,0,1,0,1,0
6,3cnienssup.ufrn.br,19,19,0,0,0,0,-0.404,-0.404,0.000,12.500,0,50,4,0,6,2
7,profletras.ufrn.br,-2,0,-4,0,0,0,0.002,0.008,-0.036,-0.036,0,0,59,0,57,0
8,mat.ufrn.br,0,0,-4,0,0,0,0.000,0.126,0.000,0.000,0,0,8,0,7,0
9,ria.ufrn.br,1,0,0,0,6,6,0.000,0.000,5.263,5.263,0,0,14,0,13,0


In [9]:
figura(p, f"Cópia local original × corrigida (posterior, n = {len(p)} sites)", "validacao-antes-depois-posterior.png")

figura: C:\Users\Mateo\Documents\tcc-ufrn-web-audit\docs\resultados\figuras\validacao-antes-depois-posterior.png


## 6. Censo dos elegíveis (análise secundária)

**Não confirmatória.** Pré-registro: adendo do censo ao ADR 004, commitado antes das medições (`1b072fd`). Universo: todos os elegíveis da seção 2 do ADR, sem a janela nem o limite por grupo, menos 14 sites fora do escopo (login, ferramenta, sem conteúdo). Regras da S18b (`v1.0.1-regras`) com `--llm`. Os 12 sites da amostra entram com a medição da S18b. Mesma família de testes da seção 1. Sites sem execução válida numa variante saem do teste do desfecho afetado (delta ausente).

In [10]:
c = pd.read_csv(CENSO)
lista = pd.read_csv(LISTA_CENSO)
faltando_censo = sorted(set(lista.id) - set(c.id))
print("sites no censo:", len(lista), "| validados:", len(c), "| faltando:", faltando_censo)
print("por grupo:", c.grupo.value_counts().to_dict())
print("deltas ausentes por desfecho:", {k: int(c[f"{k}_delta"].isna().sum()) for k in PRINCIPAIS})
print("sites com correção da LLM:", int((c.correcoes_llm > 0).sum()))
censo = familia(c)
censo.to_csv(TAB / "validacao-wilcoxon-censo.csv", index=False)
censo.round(4)

sites no censo: 134 | validados: 134 | faltando: []
por grupo: {'estatico': 71, 'wordpress': 35, 'framework-js': 19, 'outro': 9}
deltas ausentes por desfecho: {'nota_performance': 0, 'lcp_ms': 0, 'cls': 0, 'tbt_ms': 0, 'indice_semantica': 0}
sites com correção da LLM: 46


,desfecho,melhor_se,mediana_original,mediana_corrigido,n_pares,n_efetivo,melhorou,piorou,iguais,mediana_delta,W,p,r_rank_bisserial,p_holm,significativo
0,nota_performance,maior,65.5000,68.0000,134,72,52,20,62,0.0000,833.5,0.0066,0.3657,0.0197,True
1,lcp_ms,menor,9604.8500,8954.3714,134,134,106,28,0,-442.2258,1228.0,0.0000,-0.7285,0.0000,True
2,cls,menor,0.0075,0.0079,134,50,28,22,84,0.0000,578.0,0.5657,-0.0933,0.5657,False
3,tbt_ms,menor,38.5000,36.0000,134,98,56,42,36,0.0000,2106.0,0.2575,-0.1317,0.5151,False
4,indice_semantica,maior,64.5854,70.8387,134,65,63,2,69,0.0000,24.0,0.0000,0.9776,0.0000,True


### 6.1 Descritivas por grupo de tecnologia

Sem teste por grupo (`outro` tem n = 9, adendo do censo). Mediana do delta e contagem de sites que melhoraram/pioraram em cada desfecho principal.

In [11]:
linhas = []
for g, sub in c.groupby("grupo"):
    for desf, melhor in PRINCIPAIS.items():
        d = sub[f"{desf}_delta"].dropna()
        sinal = 1 if melhor == "maior" else -1
        linhas.append({"grupo": g, "desfecho": desf, "n": len(d), "mediana_delta": d.median(),
                       "melhorou": int((sinal * d > 0).sum()), "piorou": int((sinal * d < 0).sum()), "iguais": int((d == 0).sum())})
por_grupo = pd.DataFrame(linhas)
por_grupo.to_csv(TAB / "validacao-censo-por-grupo.csv", index=False)
por_grupo.round(3)

,grupo,desfecho,n,mediana_delta,melhorou,piorou,iguais
0,estatico,nota_performance,71,0.000,29,11,31
1,estatico,lcp_ms,71,-449.718,57,14,0
2,estatico,cls,71,0.000,16,14,41
3,estatico,tbt_ms,71,0.000,27,22,22
4,estatico,indice_semantica,71,5.263,38,1,32
5,framework-js,nota_performance,19,0.000,5,6,8
6,framework-js,lcp_ms,19,-18.932,11,8,0
7,framework-js,cls,19,0.000,5,1,13
8,framework-js,tbt_ms,19,0.000,9,8,2
9,framework-js,indice_semantica,19,5.263,12,0,7


In [12]:
desc_censo = pd.DataFrame([{
    "desfecho": k,
    "n": int(c[f"{k}_delta"].notna().sum()),
    "mediana_original": c[f"{k}_original"].median(),
    "mediana_corrigido": c[f"{k}_corrigido"].median(),
    "mediana_delta": c[f"{k}_delta"].median(),
    "delta_positivo": int((c[f"{k}_delta"] > 0).sum()),
    "delta_negativo": int((c[f"{k}_delta"] < 0).sum()),
    "delta_zero": int((c[f"{k}_delta"] == 0).sum()),
} for k in outros])
desc_censo.to_csv(TAB / "validacao-censo-descritivas.csv", index=False)
r = c[["id", "nota_performance_delta", "amplitude_original", "amplitude_corrigido"]].copy()
r["amplitude_max"] = r[["amplitude_original", "amplitude_corrigido"]].max(axis=1)
print("censo: sites com |delta perf| acima da maior amplitude:", int((r.nota_performance_delta.abs() > r.amplitude_max).sum()), "de", int(r.nota_performance_delta.notna().sum()))
desc_censo.round(3)

censo: sites com |delta perf| acima da maior amplitude: 23 de 134


,desfecho,n,mediana_original,mediana_corrigido,mediana_delta,delta_positivo,delta_negativo,delta_zero
0,nota_accessibility,134,80.000,85.000,0.000,52,0,82
1,nota_best_practices,134,92.000,92.000,0.000,4,0,130
2,nota_seo,134,82.500,83.000,0.000,2,0,132
3,fcp_ms,134,3302.229,3366.785,0.060,69,65,0
4,speed_index_ms,134,3533.881,3495.737,-0.384,61,73,0
5,peso_total_bytes,134,1691507.500,1600800.000,8.500,77,56,1
6,requisicoes,134,43.500,37.000,0.000,4,53,77
7,indice_semantica_pesos_iguais,134,69.647,75.093,0.000,63,2,69
8,sub_landmarks,134,75.000,75.000,0.000,45,1,88
9,sub_titulos,134,50.000,50.000,0.000,0,0,134


In [13]:
figura(c, f"Cópia local original × corrigida (censo, n = {len(c)} sites, análise secundária)", "validacao-antes-depois-censo.png")

figura: C:\Users\Mateo\Documents\tcc-ufrn-web-audit\docs\resultados\figuras\validacao-antes-depois-censo.png


In [14]:
meta["saidas"] = ["docs/resultados/tabelas/validacao-por-site.csv", "docs/resultados/tabelas/validacao-wilcoxon.csv",
                  "docs/resultados/tabelas/validacao-descritivas.csv", "docs/resultados/figuras/validacao-antes-depois.png",
                  "docs/resultados/tabelas/validacao-wilcoxon-posterior.csv", "docs/resultados/tabelas/validacao-v1-0-x-posterior.csv",
                  "docs/resultados/figuras/validacao-antes-depois-posterior.png",
                  "docs/resultados/tabelas/validacao-wilcoxon-censo.csv", "docs/resultados/tabelas/validacao-censo-por-grupo.csv",
                  "docs/resultados/tabelas/validacao-censo-descritivas.csv", "docs/resultados/figuras/validacao-antes-depois-censo.png"]
(TAB / "validacao-meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
meta["saidas"]

['docs/resultados/tabelas/validacao-por-site.csv',
 'docs/resultados/tabelas/validacao-wilcoxon.csv',
 'docs/resultados/tabelas/validacao-descritivas.csv',
 'docs/resultados/figuras/validacao-antes-depois.png',
 'docs/resultados/tabelas/validacao-wilcoxon-posterior.csv',
 'docs/resultados/tabelas/validacao-v1-0-x-posterior.csv',
 'docs/resultados/figuras/validacao-antes-depois-posterior.png',
 'docs/resultados/tabelas/validacao-wilcoxon-censo.csv',
 'docs/resultados/tabelas/validacao-censo-por-grupo.csv',
 'docs/resultados/tabelas/validacao-censo-descritivas.csv',
 'docs/resultados/figuras/validacao-antes-depois-censo.png']